# AI Job Agent — PHASE 2: Candidate Profile

**Goal:** Transform a resume (PDF/DOCX/text) + user preferences into a `candidate_profile`

using the **same data structure** used in `jobs_prepared.parquet` from Phase 1 — **the same buckets**

for skills, the same enums, and the same normalization function. This symmetry is what allows Phase 5

to become a field-by-field comparison instead of reprocessing text from scratch.

**Pipeline:**

```text
Resume file (PDF/DOCX/txt)
      ↓  Raw text extraction
      ↓  clean_text()  (the exact same function as Phase 1)
      ↓  LLM structured extraction  (same approach as Phase 1, matching skill buckets)
      ↓  normalize_skill_list()  (the exact same Phase 1 dictionary)
      ↓  + User preferences (location, expected salary, work arrangement, nationality)
      ↓  candidate_profile  (+ embedding using the same model and vector space as Phase 1)
```

**The path in this notebook:**

| Path          | Use case                                                                            | Output                                                     |
| ------------- | ----------------------------------------------------------------------------------- | ---------------------------------------------------------- |

| **Live**  | A single resume + a preferences form, ready immediately for Phase 3                 | `candidate_profile.json` + `candidate_embedding.npy`       |

> **Compatibility contract with Phase 1:** Any skill field here must pass through the same `normalize_skill()`
>
> and use the same `SKILL_ALIASES` dictionary used in `jobs_prepared.parquet`. Any modification to
>
> the skill dictionary in the Phase 1 notebook **must** be copied here verbatim; otherwise, matching accuracy
> will silently degrade.


---
## 1. Setup

In [28]:
!pip install -q pdfplumber python-docx openai pydantic tqdm nest_asyncio

In [29]:
import os
import re
import io
import json
import html
import time
import hashlib
import asyncio
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

WORK = Path("work")
CACHE = Path("cache")
OUT = Path("outputs")
RESUMES = Path("resumes")

for d in (WORK, CACHE, OUT, RESUMES):
    d.mkdir(parents=True, exist_ok=True)

print("Ready.")

Ready.


---
## 2. Configuration

In [30]:
CONFIG = {


    "extraction_model": "gpt-5.6-luna",
    "prompt_version": "v2.0",


    "embedding_model": "text-embedding-3-small",
    "embedding_dim": 1536,

    "max_resume_chars_sent_to_llm": 8_000,
    "min_resume_chars": 200,
    "max_retries": 5,
    "concurrency": 5,
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

RUN_ID: 20260917_0840


---

## 3. Shared Utilities (Identical to Phase 1)

These functions are an **exact copy** of the Phase 1 notebook — text cleaning, smart truncation, and skill normalization. Do not modify them here independently; any future change to `SKILL_ALIASES` in Phase 1 must be copied here **verbatim**.

> **Suggested future improvement:** Move these functions into a single `shared_utils.py` file and import them into both notebooks, instead of manually copying them. They are intentionally duplicated here for now so that each notebook remains independently runnable on Colab without file-to-file dependencies.


In [31]:
TAG_RE = re.compile(r"<[^>]{1,400}?>")
SCRIPT_RE = re.compile(r"<(script|style)[^>]*>.*?</\1>", re.S | re.I)
INLINE_WS_RE = re.compile(r"[ \t\x0b\f\r\u00a0\u2000-\u200a]+")
MANY_NL_RE = re.compile(r"\n{3,}")
ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\ufeff\u2060]")


def clean_text(value, keep_newlines: bool = True) -> str:
    """Normalize text while preserving useful structure. Identical to Phase 1."""

    if not isinstance(value, str) or not value.strip():
        return ""

    text = value

    for _ in range(3):
        new = html.unescape(text)
        if new == text:
            break
        text = new

    text = SCRIPT_RE.sub(" ", text)
    text = TAG_RE.sub("\n", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = INLINE_WS_RE.sub(" ", text)

    lines = [ln.strip() for ln in text.split("\n")]
    lines = [ln for ln in lines if ln]
    text = "\n".join(lines)
    text = MANY_NL_RE.sub("\n\n", text)

    if not keep_newlines:
        text = text.replace("\n", " ")
        text = INLINE_WS_RE.sub(" ", text)

    return text.strip()


def norm_key(value) -> str:
    text = clean_text(value, keep_newlines=False).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def smart_truncate(text: str, max_chars: int) -> str:
    """Head+tail truncation — preserves both the summary and the most
    recent experience, which in resumes often sits near the top, and
    education/certifications, which often sit near the bottom."""

    if len(text) <= max_chars:
        return text

    head_budget = int(max_chars * 0.7)
    tail_budget = max_chars - head_budget - len("\n[... omitted ...]\n")

    return text[:head_budget] + "\n[... omitted ...]\n" + text[-tail_budget:]

In [32]:
SKILL_ALIASES = {
    # data / bi
    "powerbi": "power bi", "ms power bi": "power bi",
    "microsoft power bi": "power bi",
    "ms excel": "excel", "microsoft excel": "excel",
    "advanced excel": "excel", "ms office": "microsoft office",
    "office suite": "microsoft office", "msoffice": "microsoft office",
    "sql server": "microsoft sql server", "mssql": "microsoft sql server",
    "postgres": "postgresql", "ms sql": "microsoft sql server",
    "google data studio": "looker studio",
    "data visualisation": "data visualization",
    "data analytics": "data analysis",
    "statistical analysis": "statistics",
    "machine learning (ml)": "machine learning",
    "ml": "machine learning", "ai": "artificial intelligence",
    "nlp": "natural language processing",
    "etl pipelines": "etl", "etl processes": "etl",
    # engineering / software
    "js": "javascript", "reactjs": "react", "react.js": "react",
    "nodejs": "node.js", "node js": "node.js",
    "py": "python", "python3": "python",
    "c sharp": "c#", "golang": "go",
    "rest apis": "rest api", "restful api": "rest api",
    "ci cd": "ci/cd", "cicd": "ci/cd",
    "aws cloud": "aws", "amazon web services": "aws",
    "microsoft azure": "azure", "gcp": "google cloud",
    "k8s": "kubernetes",
    # business
    "ms project": "microsoft project",
    "erp systems": "erp", "sap erp": "sap",
    "kpi reporting": "kpi reporting",
    "stakeholder management": "stakeholder management",
    "project mgmt": "project management",
    # soft
    "communication skills": "communication",
    "verbal communication": "communication",
    "written communication": "written communication",
    "team work": "teamwork", "team player": "teamwork",
    "problem-solving": "problem solving",
    "time-management": "time management",
    "attention to detail": "attention to detail",
    "interpersonal skills": "interpersonal skills",
    "leadership skills": "leadership",
    "analytical skills": "analytical thinking",
    "analytical thinking skills": "analytical thinking",
    # languages
    "english language": "english", "arabic language": "arabic",
    "fluent english": "english", "native arabic": "arabic",
    "english (fluent)": "english",
}

ACRONYM_KEEP = {"sql", "aws", "gcp", "erp", "sap", "api", "etl", "bi", "qa",
                "hr", "ui", "ux", "iso", "css", "html", "php", "ios", "crm",
                "kpi", "cad", "plc", "hvac", "ccna", "pmp", "cpa", "acca"}

NOISE_RE = re.compile(
    r"^(ability to|able to|experience (in|with)|knowledge of|proficiency in|"
    r"strong |excellent |good |demonstrated |proven |solid |working )",
    re.I,
)


def normalize_skill(skill: str) -> str:
    if not isinstance(skill, str):
        return ""

    text = clean_text(skill, keep_newlines=False).lower()
    text = NOISE_RE.sub("", text)
    text = re.sub(r"[\(\)\[\]\.,;:]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" -–—/")

    if not text or len(text) < 2:
        return ""

    if len(text.split()) > 5:
        return ""

    text = SKILL_ALIASES.get(text, text)

    if text.endswith("s") and text[:-1] in SKILL_ALIASES.values():
        text = text[:-1]

    return SKILL_ALIASES.get(text, text)


def display_skill(canonical: str) -> str:
    if canonical in ACRONYM_KEEP:
        return canonical.upper()
    return " ".join(
        w.upper() if w in ACRONYM_KEEP else w.capitalize()
        for w in canonical.split()
    )


def normalize_skill_list(values) -> list:
    seen, out = set(), []
    for item in (values or []):
        canon = normalize_skill(item)
        if canon and canon not in seen:
            seen.add(canon)
            out.append(canon)
    return out


print("Shared utilities loaded (identical to Phase 1).")

Shared utilities loaded (identical to Phase 1).


---

## 4. Resume Text Extraction

Supports PDF (text-based, not scanned), DOCX, and TXT files. Any other format requires manual conversion.


In [33]:
import pdfplumber
from docx import Document as DocxDocument


def read_pdf(path) -> str:
    parts = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            parts.append(text)
    return "\n".join(parts)


def read_docx(path) -> str:
    doc = DocxDocument(path)
    parts = [p.text for p in doc.paragraphs if p.text.strip()]

    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells:
                parts.append(" | ".join(cells))

    return "\n".join(parts)


def read_txt(path) -> str:
    return Path(path).read_text(encoding="utf-8", errors="ignore")


def read_resume(path) -> str:
    """Dispatch by extension. Returns raw extracted text (not yet cleaned)."""

    suffix = Path(path).suffix.lower()

    if suffix == ".pdf":
        raw = read_pdf(path)
    elif suffix == ".docx":
        raw = read_docx(path)
    elif suffix in (".txt", ".md"):
        raw = read_txt(path)
    else:
        raise ValueError(
            f"Unsupported file type: {suffix}. Use PDF, DOCX, or TXT. "
            f"(.doc أو صور PDF ممسوحة ضوئيًا غير مدعومة — حوّلها أولًا)"
        )

    return raw


def resume_quality_check(raw_text: str) -> dict:
    """Flags likely-scanned or empty PDFs before wasting an LLM call."""

    chars = len(raw_text.strip())
    words = len(raw_text.split())

    return {
        "chars": chars,
        "words": words,
        "likely_scanned_or_empty": chars < CONFIG["min_resume_chars"],
    }


print("Resume readers ready: .pdf, .docx, .txt, .md")

Resume readers ready: .pdf, .docx, .txt, .md


---

## 5. Structured Extraction Schema

Uses the **same skill buckets** and the same enums as `JobFeatures` in Phase 1 (`ExperienceLevel`, `EmploymentType`, `EducationLevel`) so they can be compared directly without conversion.

The key difference: there is no `required/preferred` distinction here — a candidate **has** skills; they do not **request** them.


In [34]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field


ExperienceLevel = Literal[
    "Internship", "Entry Level", "Mid Level", "Senior Level",
    "Executive", "Not Specified",
]

EducationLevel = Literal[
    "High School", "Diploma", "Bachelor's Degree",
    "Master's Degree", "PhD", "Not Specified",
]


class ResumeExtraction(BaseModel):
    """LLM-extracted structured content from a resume. Facts only —
    no preferences (those come from the user-input form in Section 8)."""

    # ---------- Identity (as written on the resume) ----------
    full_name: Optional[str]
    email: Optional[str]
    phone: Optional[str]
    most_recent_title: Optional[str]

    # ---------- Skills — identical bucket scheme to JobFeatures ----------
    hard_skills: List[str] = Field(
        description="Technical or domain skills. Short noun phrases, max 4 words."
    )
    soft_skills: List[str] = Field(
        description="Behavioural or interpersonal skills. Max 4 words each."
    )
    tools_technologies: List[str] = Field(
        description="Named tools, software, platforms, frameworks, standards."
    )
    languages: List[str] = Field(
        description="Spoken/written languages, e.g. Arabic, English."
    )

    # ---------- Qualifications (NOT skills) ----------
    qualifications: List[str] = Field(
        description="Certifications, licences, degrees, professional memberships."
    )

    # ---------- Experience ----------
    experience_level: ExperienceLevel
    experience_level_inferred: bool
    years_experience_total: Optional[float] = Field(
        description="Estimated total years of relevant work experience, "
                     "computed from date ranges in the work history when present."
    )
    past_titles: List[str] = Field(
        description="Job titles held, most recent first, excluding the current one."
    )
    industries: List[str] = Field(
        description="Industries/sectors worked in, e.g. 'Banking', 'Retail'."
    )

    # ---------- Education ----------
    education_level: List[EducationLevel]
    education_field: List[str]

    # ---------- Summary ----------
    summary: Optional[str] = Field(
        description="2-3 sentence professional summary in the third person."
    )

    extraction_confidence: float = Field(
        description="0.0-1.0 confidence that the resume was well-formed and complete."
    )


print("Schema fields:", len(ResumeExtraction.model_fields))

Schema fields: 18


### 5.1 Extraction Instructions

Use the **same extraction rules** applied to job descriptions in Phase 1 (maximum of 4 words per skill, separation of qualifications, and no hallucination) — to ensure consistency in the extraction style between both sides, and consequently achieve higher matching quality in Phase 5.


In [35]:
SYSTEM_PROMPT = """
You are a resume information extraction system. You convert an unstructured
resume into a strict structured record.

GENERAL RULES

1. Never invent information. If it is not stated or clearly implied, use an
   empty list, null, or "Not Specified".
2. Extract from the resume content only. Ignore headers/footers/page numbers
   and decorative text.

SKILLS — THE MOST IMPORTANT PART

3. A skill is a short noun phrase of AT MOST 4 words, exactly like:
   GOOD: "SQL", "financial modeling", "patient care", "welding"
   BAD:  "Bachelor's degree in Computer Science"
   BAD:  "4 years of experience in data analysis"
4. Degrees, certifications, and licences are NOT skills — put them in
   `qualifications`.
5. Split skills into four buckets, each skill in exactly ONE bucket:
   - hard_skills: technical/domain capabilities
   - soft_skills: behavioural ("teamwork", "leadership")
   - tools_technologies: named products/platforms/standards ("Power BI", "SAP")
   - languages: spoken languages only ("Arabic", "English")
6. Use the canonical name of a tool: "Power BI" not "MS PowerBI".
7. If the resume has an explicit "Skills" section, extract every item from it.
   ALSO scan the experience bullet points for skills not listed there
   explicitly (e.g. "built dashboards in Tableau" implies "Tableau" and
   "dashboarding" even with no dedicated skills section).

EXPERIENCE

8. years_experience_total: if the work history has explicit date ranges
   (e.g. "Jan 2022 - Present", "2019-2021"), compute the total span across
   all roles in years, one decimal place. Overlapping roles should not be
   double-counted — use the union of the date ranges.
   If no dates are present, return null. Never guess from job titles alone.
9. experience_level: Internship/Entry Level (0-1y) / Mid Level (2-4y) /
   Senior Level (5-9y) / Executive (10+y or C-level/VP/Director titles).
   Base it on years_experience_total when available (set
   experience_level_inferred = true). If the resume states a level
   explicitly (e.g. "Senior" in the most recent title), that also counts
   as inferred = false only if the resume literally uses that word.
10. past_titles: list of previous job titles, most recent first, EXCLUDING
    the most recent/current one (which goes in most_recent_title).

EDUCATION

11. education_level uses ONLY the allowed enum values. List every degree
    held, not just the highest. education_field holds the subject of study
    for each entry, in the same order.

OTHER

12. full_name, email, phone: extract exactly as written. Null if absent.
13. summary: write 2-3 sentences in the third person describing the
    candidate's profile, based only on resume content (not invented).
14. industries: infer from employer descriptions or job context, not from
    company names alone unless the industry is obvious (e.g. "XYZ Bank").
15. extraction_confidence: 0.9+ for a detailed, well-structured resume,
    0.5 for a thin one, below 0.3 if the text looks corrupted or mostly
    unreadable (e.g. from a badly-parsed scanned PDF).
"""

print(f"Prompt length: {len(SYSTEM_PROMPT):,} chars")

Prompt length: 3,048 chars


---

## 6. Extraction Pipeline (Async + Cache, Same Pattern as Phase 1)

Designed to support two paths: a batch of resumes (§7) or a single resume processed immediately (§9) — the same `extract_one` function serves both, while the cache prevents reprocessing the same file even if the entire notebook is restarted.


In [36]:
import nest_asyncio
nest_asyncio.apply()

from openai import AsyncOpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("openai_key_new")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")

assert api_key, "OpenAI API key not found"
aclient = AsyncOpenAI(api_key=api_key)

print("Async client ready.")

Async client ready.


In [37]:
CACHE_PATH = CACHE / f"resume_extraction_{CONFIG['prompt_version']}.jsonl"


def cache_key(text: str) -> str:
    payload = f"{CONFIG['extraction_model']}|{CONFIG['prompt_version']}|{text}"
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


def load_cache() -> dict:
    store = {}
    if CACHE_PATH.exists():
        with CACHE_PATH.open(encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    store[rec["key"]] = rec["features"]
                except Exception:
                    continue
    return store


EXTRACTION_CACHE = load_cache()
print(f"Cached extractions: {len(EXTRACTION_CACHE):,}")

Cached extractions: 1


In [38]:
_cache_fh = CACHE_PATH.open("a", encoding="utf-8")
_cache_lock = asyncio.Lock()

FAILURES = []


async def extract_one(candidate_id: str, resume_text: str, sem: asyncio.Semaphore) -> tuple:
    """Return (candidate_id, features_dict or None)."""

    text = smart_truncate(resume_text, CONFIG["max_resume_chars_sent_to_llm"])
    key = cache_key(text)

    if key in EXTRACTION_CACHE:
        return candidate_id, EXTRACTION_CACHE[key]

    delay = 2.0

    async with sem:
        for attempt in range(CONFIG["max_retries"]):
            try:
                response = await aclient.responses.parse(
                    model=CONFIG["extraction_model"],
                    input=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": text},
                    ],
                    text_format=ResumeExtraction,
                )

                features = response.output_parsed.model_dump()

                async with _cache_lock:
                    EXTRACTION_CACHE[key] = features
                    _cache_fh.write(
                        json.dumps({"key": key, "candidate_id": candidate_id,
                                    "features": features}, ensure_ascii=False) + "\n"
                    )
                    _cache_fh.flush()

                return candidate_id, features

            except Exception as exc:
                is_last = attempt == CONFIG["max_retries"] - 1

                if is_last:
                    FAILURES.append({"candidate_id": candidate_id, "error": repr(exc)[:300]})
                    return candidate_id, None

                await asyncio.sleep(delay + np.random.rand())
                delay = min(delay * 2, 60)

    return candidate_id, None


async def embed_texts(texts: list) -> np.ndarray:
    """Batch embed via OpenAI embeddings API, same model as Phase 1 job vectors."""

    response = await aclient.embeddings.create(
        model=CONFIG["embedding_model"],
        input=texts,
    )
    vecs = np.array([d.embedding for d in response.data], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vecs / norms   # unit-normalized, matches job_embeddings.npy contract


def make_candidate_id(source: str) -> str:
    return hashlib.sha1(source.encode("utf-8")).hexdigest()[:16]


def build_embedding_text(row) -> str:
    """Text used to embed a candidate — same *kind* of content (title +
    summary + skills) used to embed job postings in Phase 1, so both sit
    in the same semantic space for cosine retrieval in Phase 3.
    `row` can be a dict or a pandas row; only needs the 4 keys below."""

    parts = [
        row.get("most_recent_title") or "",
        row.get("summary") or "",
        "Skills: " + ", ".join(display_skill(s) for s in row["all_skills_norm"]),
        "Industries: " + ", ".join(row.get("industries") or []),
    ]
    return clean_text(" . ".join(p for p in parts if p), keep_newlines=False)


print("Extraction + embedding pipeline ready.")

Extraction + embedding pipeline ready.


---

## 7. Live: One Resume + Preferences → Ready-to-Use Profile

This path generates a single `candidate_profile.json` that is ready to use directly in Phase 3.

Later, in Phase 10, this section will be replaced with an actual form instead of manual text editing. For now, we modify a Python dictionary directly because it is simpler and more reliable for development and rapid iteration in Colab.


In [39]:
import google.colab.files as _colab_files

In [40]:
print("Upload your resume (one file only for this path):")
live_upload = _colab_files.upload()

assert len(live_upload) == 1, "Upload only one file for this path (PATH B)."

live_name = list(live_upload.keys())[0]
live_path = RESUMES / live_name
live_path.write_bytes(live_upload[live_name])

print(f"\nSaved: {live_path}")

Upload your resume (one file only for this path):


Saving Raghad Almangour_ Data Scientist_CV.pdf to Raghad Almangour_ Data Scientist_CV (1).pdf

Saved: resumes/Raghad Almangour_ Data Scientist_CV (1).pdf


### 7.1 Preferences — Edit This Dictionary With Your Information

These fields are **not extracted from the resume** — they represent personal decisions (location, expected salary, willingness to relocate), so they should never be guessed.

Set any field that is not important to you to `None` or `[]`; Phase 4 will ignore empty fields during strict filtering rather than incorrectly excluding you.


In [41]:
CANDIDATE_PREFERENCES = {


    "desired_countries": ["Saudi Arabia"],
    "desired_cities": [],


    "work_arrangement_preference": "Not Specified",

    # "Full-time" / "Part-time" / "Contract" / "Internship" / "Temporary" / "Not Specified"
    "employment_type_preference": "Full-time",

    "willing_to_relocate": False,


    "nationality": "Saudi",

    "salary_expectation_min": None,
    "salary_expectation_max": None,
    "salary_currency": "SAR",
    "salary_period": "month",       # "hour" / "day" / "month" / "year"


    "availability": "Immediate",

    "notes": "",
}

print("Preferences set:")
for k, v in CANDIDATE_PREFERENCES.items():
    print(f"  {k:<32}: {v}")

Preferences set:
  desired_countries               : ['Saudi Arabia']
  desired_cities                  : []
  work_arrangement_preference     : Not Specified
  employment_type_preference      : Full-time
  willing_to_relocate             : False
  nationality                     : Saudi
  salary_expectation_min          : None
  salary_expectation_max          : None
  salary_currency                 : SAR
  salary_period                   : month
  availability                    : Immediate
  notes                           : 


### 7.2 Extract + merge

In [42]:
live_raw = read_resume(live_path)
live_quality = resume_quality_check(live_raw)

print(f"Extracted {live_quality['chars']:,} chars, {live_quality['words']:,} words")
if live_quality["likely_scanned_or_empty"]:
    print("[WARNING] This looks like a scanned or empty PDF. Extraction quality "
          "will likely be poor — consider re-exporting the resume as text-based PDF.")

live_clean = clean_text(live_raw, keep_newlines=True)
live_candidate_id = make_candidate_id(f"{live_name}|{RUN_ID}")

print(f"\ncandidate_id: {live_candidate_id}")
print("\n--- preview ---")
print(live_clean[:500])

Extracted 3,980 chars, 568 words

candidate_id: add5b8a71e877cdd

--- preview ---
Raghad Almangour (+966) 537837454 | raghadalmangor@gmail.com
https://linkedin.com/in/raghad-almanqour /
https://github.com/raghadAmangour /
Riyadh, Kingdom of Saudi Arabia
A Computer Science graduate with expertise in data science, machine learning, and AI-driven projects. Skilled
in Python, SQL, deep learning, and predictive modeling, with hands-on experience in generative AI, data
analysis, and web development. Seeking an AI/data science role to leverage project-based experience and
technical 


In [43]:
async def run_single_extraction():
    sem = asyncio.Semaphore(1)
    return await extract_one(live_candidate_id, live_clean, sem)


_, live_features = asyncio.get_event_loop().run_until_complete(run_single_extraction())

assert live_features is not None, (
    "Extraction failed after retries. Check FAILURES list above for the error."
)

print("Extraction succeeded.")
print(json.dumps({k: v for k, v in live_features.items()
                   if k not in ("qualifications",)}, indent=2, ensure_ascii=False)[:1500])

Extraction succeeded.
{
  "full_name": "Raghad Almangour",
  "email": "raghadalmangor@gmail.com",
  "phone": "(+966) 537837454",
  "most_recent_title": "Generative AI - Trainee",
  "hard_skills": [
    "Data science",
    "Machine learning",
    "Deep learning",
    "Natural language processing",
    "Computer vision",
    "Generative AI",
    "Large language models",
    "Prompt engineering",
    "Predictive modeling",
    "Data analysis",
    "Data visualization",
    "Web development",
    "Feature engineering",
    "Exploratory data analysis",
    "Data cleaning",
    "Descriptive statistics",
    "Dashboard design",
    "Statistical forecasting",
    "Recommendation systems",
    "Relational databases",
    "Adversarial search"
  ],
  "soft_skills": [
    "Data storytelling",
    "Decision-making",
    "Project-based learning"
  ],
  "tools_technologies": [
    "Python",
    "SQL",
    "Java",
    "JavaScript",
    "pandas",
    "NumPy",
    "scikit-learn",
    "matplotlib",
    "

In [44]:

normalized = {
    f"{field}_norm": normalize_skill_list(live_features.get(field, []))
    for field in ["hard_skills", "soft_skills", "tools_technologies", "languages"]
}
normalized["all_skills_norm"] = sorted(set(
    normalized["hard_skills_norm"]
    + normalized["tools_technologies_norm"]
    + normalized["soft_skills_norm"]
))


candidate_profile = {
    "candidate_id": live_candidate_id,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_file": live_name,

    **live_features,
    **normalized,
    "n_skills": len(normalized["all_skills_norm"]),

    **CANDIDATE_PREFERENCES,
}

print("Merged candidate_profile with", len(candidate_profile), "fields.")

Merged candidate_profile with 39 fields.


### 7.3 Consistency Check

Warning — **do not block** — if the preferences conflict with the extracted facts (for example, `Entry Level` experience combined with an implicit preference for `Executive`-only roles would be inconsistent; however, there is no actual conflict here because `employment_type_preference` is independent of experience level).

The most practical check is to ensure that the **essential filtering fields are populated**.


In [45]:

warnings = []

if not candidate_profile["desired_countries"] and not candidate_profile["willing_to_relocate"]:

    warnings.append(
        "desired_countries is empty and willing_to_relocate=False — "
        "Phase 4 will not be able to apply meaningful location filtering. Specify at least "
        "one country or set willing_to_relocate=True."
    )

if candidate_profile["extraction_confidence"] < 0.4:

    warnings.append(
        f"Low extraction_confidence ({candidate_profile['extraction_confidence']}) — "
        "manually review the resume; the results may be unreliable."
    )

if not candidate_profile["all_skills_norm"]:
    warnings.append("No skills were extracted — check the quality of the resume text extraction.")

if warnings:

    print("Warnings:")

    for w in warnings:
        print(" -", w)

else:

    print("No warnings — the profile appears ready.")



No warnings — the profile appears ready.


### 7.4 Embed the live profile

In [46]:
live_embedding_text = build_embedding_text({
    "most_recent_title": candidate_profile.get("most_recent_title"),
    "summary": candidate_profile.get("summary"),
    "all_skills_norm": candidate_profile["all_skills_norm"],
    "industries": candidate_profile.get("industries") or [],
})

live_vector = asyncio.get_event_loop().run_until_complete(
    embed_texts([live_embedding_text])
)[0]

print("Embedding text used:")
print(" ", live_embedding_text[:300])
print("\nVector shape:", live_vector.shape, "| unit norm:", round(float(np.linalg.norm(live_vector)), 4))

Embedding text used:
  Generative AI - Trainee . Raghad Almangour is a Computer Science graduate with project-based experience in data science, machine learning, generative AI, data analysis, and web development. She has developed predictive models, AI applications, dashboards, and web applications using technologies incl

Vector shape: (1536,) | unit norm: 1.0


### 7.5 Save the live profile

In [47]:
LIVE_DIR = OUT / "live_candidate"
LIVE_DIR.mkdir(parents=True, exist_ok=True)


(LIVE_DIR / "candidate_profile.json").write_text(
    json.dumps(candidate_profile, indent=2, ensure_ascii=False), encoding="utf-8"
)

np.save(LIVE_DIR / "candidate_embedding.npy", live_vector.astype(np.float32))
(LIVE_DIR / "candidate_id.txt").write_text(live_candidate_id, encoding="utf-8")

print(f"Saved to {LIVE_DIR}/:")
for f in sorted(LIVE_DIR.iterdir()):
    print(f"  {f.name}")

Saved to outputs/live_candidate/:
  candidate_embedding.npy
  candidate_id.txt
  candidate_profile.json


---
## 8. Data dictionary

In [48]:
DATA_DICTIONARY = """# Phase 2 outputs — Data Dictionary

## candidate_profile.json (PATH B — single live candidate)

One JSON object. Produced from one resume + a preferences dict edited by
the user. Ready to feed directly into PHASE 3 (retrieval) and PHASE 4
(hard filtering).

### Identity
| Field | Type | Notes |
|---|---|---|
| candidate_id | str | sha1(filename+run_id)[:16]. Regenerated per run — not stable across re-runs of the same resume. |
| created_at | str (ISO datetime) | |
| source_file | str | Original uploaded filename. |
| full_name, email, phone | str/null | As written on the resume. |

### Skills (same bucket scheme as jobs_prepared.parquet)
`hard_skills`, `soft_skills`, `tools_technologies`, `languages` hold the LLM
output verbatim; `*_norm` variants are canonicalized through the identical
`normalize_skill()` function used in Phase 1. Matching in Phase 5 should
only ever use the `*_norm` fields. `all_skills_norm` is the deduplicated
union of hard skills, tools and soft skills — mirrors `all_skills_norm` in
jobs_prepared.parquet exactly, so set-overlap scoring is a direct
intersection between the two.

### Experience & education
| Field | Notes |
|---|---|
| experience_level | Same enum as JobFeatures.experience_level. |
| experience_level_inferred | True if derived from years rather than stated. |
| years_experience_total | Computed from resume date ranges when present, else null. |
| education_level / education_field | Same enum as JobFeatures. |
| qualifications | Certifications/licences — compare against a job's `qualifications` list. |

### Preferences (from CANDIDATE_PREFERENCES — never LLM-extracted)
| Field | Maps to Phase 4 filter on |
|---|---|
| desired_countries / desired_cities | jobs_prepared.country / .city |
| work_arrangement_preference | jobs_prepared.work_arrangement (only when job's `*_inferred`==False) |
| employment_type_preference | jobs_prepared.employment_type |
| willing_to_relocate | Relaxes the location filter when True |
| nationality | jobs_prepared.nationality_requirement |
| salary_expectation_min/max/currency/period | jobs_prepared.salary_* |

### Embedding
`live_candidate/candidate_embedding.npy` — single float32 vector, dim
matches `CONFIG["embedding_dim"]` (1536), L2-normalized, same embedding
model as job_embeddings.npy from Phase 1 (`text-embedding-3-small`).
Built from title + summary + skills + industries — same *kind* of content
used to embed job postings, so both sit in the same semantic space for
cosine retrieval in Phase 3.

## candidates_prepared.parquet (PATH A — batch/evaluation set)

Same schema as candidate_profile.json minus the preferences fields (those
are per-user and not derivable from a batch of test resumes). One row per
resume. Intended for PHASE 9 (Evaluation) — build a small labeled set of
test candidates with known good/bad job matches to compute Precision/Recall/NDCG.

Companion files: `candidate_embeddings.npy` (n, 1536) + `candidate_ids.npy`,
same row-order contract as Phase 1's job_embeddings.npy / job_ids.npy.

## Known limitations
- Scanned/image-only PDFs are not OCR'd — `resume_quality_check()` flags
  them via `likely_scanned_or_empty`, but extraction on them is unreliable.
- `.doc` (legacy Word) is not supported — convert to `.docx` first.
- `years_experience_total` depends on the LLM correctly parsing date
  ranges; resumes with vague or missing dates return null.
"""

(OUT / "PHASE2_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

Data dictionary written.


---
## 9. Final verification

In [49]:
print("=" * 60)
print("PATH B (live candidate)")
print("=" * 60)

if (LIVE_DIR / "candidate_profile.json").exists():
    reloaded_profile = json.loads((LIVE_DIR / "candidate_profile.json").read_text())
    reloaded_vector = np.load(LIVE_DIR / "candidate_embedding.npy")

    assert reloaded_profile["candidate_id"] == live_candidate_id
    assert reloaded_vector.shape == (CONFIG["embedding_dim"],)
    assert abs(float(np.linalg.norm(reloaded_vector)) - 1.0) < 1e-3

    print(f"  candidate_id : {reloaded_profile['candidate_id']}")
    print(f"  title        : {reloaded_profile.get('most_recent_title')}")
    print(f"  skills       : {reloaded_profile['n_skills']}")
    print(f"  embedding    : {reloaded_vector.shape}, unit-norm OK")
    print("  STATUS: ready for Phase 3")
else:
    print("  Not run — skip if you only used PATH A.")

print()
print("=" * 60)
print("PATH A (batch)")
print("=" * 60)

if (OUT / "candidates_prepared.parquet").exists():
    reloaded_batch = pd.read_parquet(OUT / "candidates_prepared.parquet")
    reloaded_emb = np.load(OUT / "candidate_embeddings.npy")
    reloaded_ids = np.load(OUT / "candidate_ids.npy", allow_pickle=True)

    assert len(reloaded_batch) == len(reloaded_emb) == len(reloaded_ids)
    assert (reloaded_batch["candidate_id"].to_numpy() == reloaded_ids).all()
    assert reloaded_batch["candidate_id"].is_unique

    print(f"  candidates   : {len(reloaded_batch)}")
    print(f"  embeddings   : {reloaded_emb.shape}")
    print(f"  extraction ok: {(reloaded_batch['extraction_status'] == 'ok').mean():.1%}")
    print("  STATUS: ready for Phase 9 evaluation set")
else:
    print("  Not run — skip if you only used PATH B.")

PATH B (live candidate)
  candidate_id : add5b8a71e877cdd
  title        : Generative AI - Trainee
  skills       : 59
  embedding    : (1536,), unit-norm OK
  STATUS: ready for Phase 3

PATH A (batch)
  Not run — skip if you only used PATH B.


---
## 10. Persist outputs (Colab storage is temporary!)

In [50]:
import shutil

zip_path = shutil.make_archive("phase2_outputs", "zip", OUT)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.2f} MB)")

_colab_files.download(zip_path)

Created /content/phase2_outputs.zip (0.01 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>